# IEM creativity tilt — ring GMM with a hole

The tilted density re-weights the prior: $q_\lambda(x) \propto p(x) \cdot \exp(\lambda \cdot \mathbb{E}_{x' \sim p}[D(x', x)])$, concentrating mass on points that are far (in the IEM sense) from the observed distribution — revealing the creative gap left by the missing mode.

In [ ]:
import sys, math, torch, matplotlib.pyplot as plt
sys.path.insert(0, '.')
from creativity_measure import *
dtype = torch.float64
torch.set_default_dtype(dtype)

In [ ]:
# Ring GMM with hole — pure tensor ops (vmap-safe, no torch.distributions)
N_TOTAL, HOLE_IDX, RADIUS, SIGMA = 12, 0, 4.0, 0.3
angles = 2 * math.pi * torch.arange(N_TOTAL, dtype=dtype) / N_TOTAL
all_means = torch.stack([RADIUS * torch.cos(angles), RADIUS * torch.sin(angles)], dim=-1)
mask = torch.ones(N_TOTAL, dtype=torch.bool)
mask[HOLE_IDX] = False
MEANS = all_means[mask]          # (K, 2) observed modes (hole removed)
K = MEANS.shape[0]
S2 = SIGMA ** 2
logK = math.log(K)

def log_pX(x):
    # Equal-weight isotropic GMM, pure tensor ops (vmap-safe)
    diff = x.unsqueeze(-2) - MEANS          # (..., K, 2)
    quad = diff.pow(2).sum(-1) / S2         # (..., K)
    logcomp = -0.5 * (quad + 2 * math.log(2 * math.pi * S2))
    return torch.logsumexp(logcomp, dim=-1) - logK

def log_pY(y, gamma):
    # Y = gamma*X + sqrt(gamma)*W; comp k ~ N(gamma*mu_k, (gamma^2*S2 + gamma)*I)
    var = gamma ** 2 * S2 + gamma
    diff = y.unsqueeze(-2) - gamma * MEANS  # (..., K, 2)
    quad = diff.pow(2).sum(-1) / var
    logcomp = -0.5 * (quad + 2 * torch.log(2 * math.pi * var))
    return torch.logsumexp(logcomp, dim=-1) - logK

def ring_sample(n):
    comp = torch.randint(K, (n,))
    return MEANS[comp] + SIGMA * torch.randn(n, 2, dtype=dtype)

p = Density(log_pX, log_pY, sample=ring_sample, d=2)
print(f"Ring GMM: {K} observed modes (hole at index {HOLE_IDX}), radius={RADIUS}, sigma={SIGMA}")

In [ ]:
# Grid and reference points
grid_points, XX, YY, cell_area = make_grid((-6, 6), (-6, 6), grid_n=50, dtype=dtype)
log_p_grid = p.log_p_X(grid_points)
x_refs = p.sample(32, seed=123)
print(f"Grid: {grid_points.shape[0]} points, cell_area={cell_area:.4f}")
print(f"x_refs shape: {x_refs.shape}")

In [ ]:
# Local IEM tilt — 3-panel plot
LAM = 5.0
GAMMAS_LOCAL = torch.logspace(-4, 4, 200, base=2, dtype=dtype)
D_local = LocalIEMDistance(p, GAMMAS_LOCAL, num_noises=50)

# Chunk grid in blocks of 128 to keep memory modest
BLOCK = 128
chunks = [grid_points[i:i+BLOCK] for i in range(0, grid_points.shape[0], BLOCK)]
log_q_un_local = torch.cat([
    tilted_log_density(chunk, p, D_local, x_refs, lam=LAM)
    for chunk in chunks
])

score_local = expected_distance(D_local, grid_points, x_refs)
log_q_local, q_local, Z_local = grid_normalize(log_q_un_local, cell_area)
print(f"Z_local = {Z_local.item():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_field(log_p_grid, XX, YY, ax=axes[0], title="log p(x)",
           marked=MEANS, missing=all_means[~mask])
plot_field(score_local, XX, YY, ax=axes[1], title="E[D_local(x', x)]  (local IEM score)",
           marked=MEANS, missing=all_means[~mask])
plot_field(log_q_local, XX, YY, ax=axes[2], title=f"log q_λ(x)  [local IEM, λ={LAM}]",
           marked=MEANS, missing=all_means[~mask])
fig.suptitle("Local IEM creativity tilt", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Global IEM (f_identity) tilt — 3-panel plot
GAMMAS_GLOBAL = torch.logspace(-10, 10, 200, base=2, dtype=dtype)
D_global = GlobalIEMDistance(p, GAMMAS_GLOBAL, num_eps=50, f=f_identity, verbose=True)

# Chunk grid (same pattern as the local cell) — keeps the SDE batch modest
log_q_un_global = torch.cat([
    tilted_log_density(chunk, p, D_global, x_refs, lam=LAM)
    for chunk in chunks
])

score_global = expected_distance(D_global, grid_points, x_refs)
log_q_global, q_global, Z_global = grid_normalize(log_q_un_global, cell_area)
print(f"\nZ_global (f_identity) = {Z_global.item():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_field(log_p_grid, XX, YY, ax=axes[0], title="log p(x)",
           marked=MEANS, missing=all_means[~mask])
plot_field(score_global, XX, YY, ax=axes[1], title="E[D_global(x', x)]  (global IEM score)",
           marked=MEANS, missing=all_means[~mask])
plot_field(log_q_global, XX, YY, ax=axes[2], title=f"log q_λ(x)  [global IEM f_identity, λ={LAM}]",
           marked=MEANS, missing=all_means[~mask])
fig.suptitle("Global IEM creativity tilt (f=identity)", fontsize=13)
plt.tight_layout()
plt.show()

Use `f=f_square` in `GlobalIEMDistance` for the quadratic generalized IEM distance (weights the integrand by $(2\alpha z_\gamma)^2$), and `EuclideanDistance()` as a simple Euclidean-distance baseline for `D`.